In [1]:
pip install ESM

  Using cached esm-3.2.1.post1-py3-none-any.whl.metadata (18 kB)
  Using cached torchtext-0.6.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached transformers-4.48.1-py3-none-any.whl.metadata (44 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached msgpack_numpy-0.4.8-py2.py3-none-any.whl.metadata (5.0 kB)
  Using cached cloudpathlib-0.23.0-py3-none-any.whl.metadata (16 kB)
  Using cached tenacity-9.1.4-py3-none-any.whl.metadata (1.2 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.21.4-cp39-abi3-macosx_11_0_arm64.whl.metadata (6.7 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached hf_xet-1.4.2-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 1.1 MB/s  0:00:03 eta 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) .

In [4]:
import os
import numpy as np
import pandas as pd

class PDBparser:
    
    def __init__(self, pdbPath):
        self.__pdbPath = pdbPath

    def get_residues(self):
        with open(self.__pdbPath, 'r') as file:
            aa_map = {
                'ALA': 'A', 'ARG': 'R', 'ASN': 'N', 'ASP': 'D',
                'CYS': 'C', 'GLN': 'Q', 'GLU': 'E', 'GLY': 'G',
                'HIS': 'H', 'ILE': 'I', 'LEU': 'L', 'LYS': 'K',
                'MET': 'M', 'PHE': 'F', 'PRO': 'P', 'SER': 'S',
                'THR': 'T', 'TRP': 'W', 'TYR': 'Y', 'VAL': 'V'}
            
            seen_residues = {}

            for line in file:
                line = line.strip()

                if line.startswith('ATOM'):
                    res_name = line[17:20]
                    chain = line[21]
                    res_num = int(line[22:26])

                    if chain not in seen_residues:
                        seen_residues[chain] = {}

                    if res_num not in seen_residues[chain]:
                        seen_residues[chain][res_num] = aa_map.get(res_name, 'err')
            
            sequences = {}
            for chain, residues in seen_residues.items():
                sequences[chain] = "".join(residues[num] for num in sorted(residues))     

        df = pd.DataFrame(
            list(sequences.items()),
            columns=['chain', 'sequence']
        )

        return df


pdb = PDBparser("/Users/austingilbride/masters_sem_2/SBI/Predicting_Binding_Sites/1CY1.pdb")
pdb_parsed = pdb.get_residues()


pdb_parsed.head()

pdb_parsed = pdb_parsed.reset_index(drop=True)              # ensure stable index
pdb_parsed['seq_list'] = pdb_parsed['sequence'].apply(list) # turn sequence into list of chars

pdb_parsed_residues = (
    pdb_parsed[['chain', 'seq_list']]
    .explode('seq_list')
    .rename(columns={'seq_list': 'residue'})
    .reset_index(drop=True)
)[['chain', 'residue']]
pdb_parsed_residues.head()

,chain,residue
0,A,G
1,A,K
2,A,A
3,A,L
4,A,V


In [5]:

from esm.models.esmc import ESMC
from esm.sdk.api import ESMProtein, LogitsConfig

class ESM_df:
    """Apply ESM model to get protein embeddings in df format from a PDB sequence
    Methods:
        get_embeddings == gets us the embeddings from the ESM of a PDB file 
    """

    def __init__(self, pdb_object):
        self.pdb_object = pdb_object
        
    def get_embeddings(self, pdb_object):
        client = ESMC.from_pretrained("esmc_300m")
        os.makedirs("embeddings_proteins2", exist_ok=True)
        dictionary = {}
        for chain in pdb_object:
            sequence_full = pdb_object["sequence"]
            protein = ESMProtein(sequence=sequence_full)
            protein_tensor = client.encode(protein)
            
            with torch.no_grad():
                output = client.logits(
                protein_tensor,
                LogitsConfig(sequence=True, return_embeddings=True)
                )      
            embeddings = output.embeddings.squeeze(0).half().cpu().numpy()[1:-1, :]
            dictionary[chain] = embeddings
        
        output = pd.from_dict(dictionary)
        return output




In [6]:
embeddings = ESM_df(pdb_parsed)
df_with_embeddings = embeddings.get_embeddings(pdb_parsed)

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 24105.20it/s]


TypeError: TextEncodeInput must be Union[TextInputSequence, Tuple[InputSequence, InputSequence]]